In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load pre-trained tokenizer and model
local_model_path = r"C:\\Users\\ungdu\\Downloads\\LLM_Test\\LLM"
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AutoModelForCausalLM.from_pretrained(local_model_path)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-

In [3]:
print(model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-

In [4]:
print(model.config)

GemmaConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "C:\\\\Users\\\\ungdu\\\\Downloads\\\\LLM_Test\\\\LLM",
  "architectures": [
    "GemmaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "eos_token_id": 1,
  "head_dim": 256,
  "hidden_act": "gelu",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 16384,
  "max_position_embeddings": 8192,
  "model_type": "gemma",
  "num_attention_heads": 8,
  "num_hidden_layers": 18,
  "num_key_value_heads": 1,
  "pad_token_id": 0,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "torch_dtype": "float16",
  "transformers_version": "4.46.1",
  "unsloth_version": "2024.8",
  "use_cache": true,
  "vocab_size": 256000
}



In [6]:
# Input query
query = "quả cam ngon."

In [7]:
# Tokenize query
input_ids = tokenizer(query, return_tensors='pt').input_ids.to(device)

# Generate attention mask (1 for valid tokens, 0 for padding)
# Create a causal mask to prevent attending to future tokens
seq_len = input_ids.size(1)
causal_mask = torch.tril(torch.ones((seq_len, seq_len), device=device)).view(1, 1, seq_len, seq_len)

# Combine attention mask with causal mask
attention_mask = causal_mask

# Generate position IDs
position_ids = torch.arange(0, seq_len, dtype=torch.long, device=device).unsqueeze(0)

# Display tokens, IDs, and attention mask in a DataFrame
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
token_df = pd.DataFrame({
    'Token': tokens,
    'Token ID': input_ids[0].cpu().numpy(),
    'Attention Mask': attention_mask[0, 0, :, :].cpu().numpy().diagonal()
})
print("Tokenization and Attention Mask:")
print(token_df)

Tokenization and Attention Mask:
   Token  Token ID  Attention Mask
0  <bos>         2             1.0
1     qu       592             1.0
2      ả    235637             1.0
3   ▁cam      2554             1.0
4  ▁ngon     60774             1.0
5      .    235265             1.0


In [8]:
# Save token_df to a CSV file
token_df.to_csv('token_data.csv', index=False)
print("Data saved successfully: 'token_data.csv'")

Data saved successfully: 'token_data.csv'


In [9]:
# Retrieve token embeddings
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)

# Print embeddings and shape
print("\nEmbeddings:")
print(embeddings)
print(f"Shape of Embeddings: {embeddings.shape}")
# Prepare embeddings DataFrame
embeddings_np = embeddings.squeeze(0).cpu().numpy()  # Remove batch dimension and convert to numpy
embeddings_list = embeddings_np.tolist()  # Convert to list for inclusion in DataFrame

# Create DataFrame with embeddings and related data
embeddings_df = pd.DataFrame({
    'Token': tokens,
    'Token ID': input_ids[0].cpu().numpy(),
    'Attention Mask': attention_mask[0, 0, :, :].cpu().numpy().diagonal(),
    'Embeddings': embeddings_list
})

# Save embeddings DataFrame to a CSV file
embeddings_df.to_csv('embeddings_data.csv', index=False)

print("Data saved successfully: 'embeddings_data.csv'")


Embeddings:
tensor([[[ 0.1035,  0.0052, -0.0330,  ..., -0.0170, -0.0087, -0.0099],
         [ 0.2344,  0.0259, -0.0623,  ..., -0.0483,  0.0439, -0.0269],
         [ 0.1855, -0.0294, -0.0781,  ..., -0.0306,  0.0396,  0.0098],
         [ 0.2812, -0.0095,  0.0081,  ...,  0.0386,  0.1367, -0.0913],
         [ 0.2471,  0.0349,  0.0234,  ..., -0.0043,  0.2266, -0.0304],
         [ 0.1836, -0.0625, -0.1787,  ..., -0.0371, -0.0052, -0.0171]]])
Shape of Embeddings: torch.Size([1, 6, 2048])
Data saved successfully: 'embeddings_data.csv'


In [10]:
# Pass through Transformer layers
output = embeddings
print(f"Initial Embedding Output: {output}, Shape: {output.shape}")
for i, layer in enumerate(model.model.layers):
    print(f"\n--- Layer {i + 1} ---")
    
    # Input Layer Normalization
    normed_input = layer.input_layernorm(output)
    print(f"Input Layer Normalization Output: {normed_input}, Shape: {normed_input.shape}")
    
    # Self-Attention Layer
    print(f"--- Self-Attention Components ---")
    print(f"Query Projection Weights: {layer.self_attn.q_proj.weight}, Shape: {layer.self_attn.q_proj.weight.shape}")
    print(f"Key Projection Weights: {layer.self_attn.k_proj.weight}, Shape: {layer.self_attn.k_proj.weight.shape}")
    print(f"Value Projection Weights: {layer.self_attn.v_proj.weight}, Shape: {layer.self_attn.v_proj.weight.shape}")
    print(f"Output Projection Weights: {layer.self_attn.o_proj.weight}, Shape: {layer.self_attn.o_proj.weight.shape}")
    
    attn_output = layer.self_attn(
        normed_input,
        attention_mask=attention_mask,  # Provide attention mask
        position_ids=position_ids,      # Provide position IDs
        past_key_value=None,            # No caching for the first pass
        output_attentions=False         # Avoid outputting attention weights
    )
    print(f"Self-Attention Output: {attn_output[0]}, Shape: {attn_output[0].shape}")
    
    # Add residual connection
    output = output + attn_output[0]
    print(f"Output after Residual Connection (Self-Attention): {output}, Shape: {output.shape}")
    
    # Post-Attention Normalization
    normed_attn_output = layer.post_attention_layernorm(output)
    print(f"Post-Attention Normalization Output: {normed_attn_output}, Shape: {normed_attn_output.shape}")
    
    # Feed-Forward Layer
    print(f"--- Feed-Forward Components ---")
    print(f"Gate Projection Weights: {layer.mlp.gate_proj.weight}, Shape: {layer.mlp.gate_proj.weight.shape}")
    print(f"Up Projection Weights: {layer.mlp.up_proj.weight}, Shape: {layer.mlp.up_proj.weight.shape}")
    print(f"Down Projection Weights: {layer.mlp.down_proj.weight}, Shape: {layer.mlp.down_proj.weight.shape}")
    
    ff_output = layer.mlp(normed_attn_output)
    # print(f"Feed-Forward Output: {ff_output}, Shape: {ff_output.shape}")
    
    # Add residual connection
    output = output + ff_output
    print(f"Output after Residual Connection (Feed-Forward): {output}, Shape: {output.shape}")

Initial Embedding Output: tensor([[[ 0.1035,  0.0052, -0.0330,  ..., -0.0170, -0.0087, -0.0099],
         [ 0.2344,  0.0259, -0.0623,  ..., -0.0483,  0.0439, -0.0269],
         [ 0.1855, -0.0294, -0.0781,  ..., -0.0306,  0.0396,  0.0098],
         [ 0.2812, -0.0095,  0.0081,  ...,  0.0386,  0.1367, -0.0913],
         [ 0.2471,  0.0349,  0.0234,  ..., -0.0043,  0.2266, -0.0304],
         [ 0.1836, -0.0625, -0.1787,  ..., -0.0371, -0.0052, -0.0171]]]), Shape: torch.Size([1, 6, 2048])

--- Layer 1 ---
Input Layer Normalization Output: tensor([[[ 0.0000,  0.0665, -0.1686,  ..., -0.1797, -0.0977, -0.1152],
         [ 0.0000,  0.9540, -0.9272,  ..., -1.4906,  1.4414, -0.9109],
         [ 0.0000, -1.0023, -1.0753,  ..., -0.8732,  1.1989,  0.3081],
         [ 0.0000, -0.3092,  0.1057,  ...,  1.0479,  3.9503, -2.7284],
         [ 0.0000,  1.1445,  0.3104,  ..., -0.1188,  6.6084, -0.9169],
         [ 0.0000, -2.8512, -3.2936,  ..., -1.4161, -0.2106, -0.7174]]],
       grad_fn=<MulBackward0>), Sh

In [11]:
# Final Normalization
print(f"--- Final Layer Normalization ---")
print(f"Final Layer Normalization Weights: {model.model.norm.weight}, Shape: {model.model.norm.weight.shape}")
if hasattr(model.model.norm, 'bias'):
    print(f"Final Layer Normalization Bias: {model.model.norm.bias}, Shape: {model.model.norm.bias.shape}")
else:
    print("Final Layer Normalization does not have a bias attribute.")

final_normed_output = model.model.norm(output)
print(f"Final Normalized Output: {final_normed_output}, Shape: {final_normed_output.shape}")

# Language Modeling Head
# print(f"--- Language Modeling Head ---")
print(f"LM Head Weights: {model.lm_head.weight}, Shape: {model.lm_head.weight.shape}")

--- Final Layer Normalization ---
Final Layer Normalization Weights: Parameter containing:
tensor([0.1572, 0.6172, 0.4609,  ..., 0.6094, 0.3789, 0.6445],
       requires_grad=True), Shape: torch.Size([2048])
Final Layer Normalization does not have a bias attribute.
Final Normalized Output: tensor([[[-2.4780,  0.8109, -0.8696,  ...,  0.1381, -0.5261,  0.0291],
         [-2.5911,  0.8387, -0.9246,  ...,  0.5105, -0.4217,  0.2683],
         [-1.9209,  0.8104,  0.3499,  ...,  0.7126, -0.6071, -0.7112],
         [-2.5128,  0.9827, -0.3958,  ...,  0.2897, -0.8508, -0.1213],
         [-2.5959,  0.5184, -0.3973,  ...,  0.6602, -0.2726, -0.3047],
         [-2.4241,  1.1437,  0.1856,  ..., -0.5417, -0.0118, -0.6800]]],
       grad_fn=<MulBackward0>), Shape: torch.Size([1, 6, 2048])
LM Head Weights: Parameter containing:
tensor([[ 5.2344e-01, -3.5889e-02,  5.9814e-02,  ...,  7.7637e-02,
          2.3535e-01,  3.8330e-02],
        [ 1.5137e-01, -1.4453e-01, -1.1719e-01,  ..., -1.9409e-02,
        

In [12]:
# Language Modeling Head
logits = model.lm_head(final_normed_output)
probs = torch.softmax(logits, dim=-1)

# Get probabilities for the token "▁ngon"
word_index = tokens.index("▁ngon") if "▁ngon" in tokens else -1
if word_index != -1:
    word_probs = probs[0, word_index]
    print(f"Probabilities for token '▁ngon': {word_probs}")
else:
    print("Token '▁ngon' not found in the input query.")

Probabilities for token '▁ngon': tensor([0.0000e+00, 5.5457e-10, 2.3195e-24,  ..., 2.9708e-43, 0.0000e+00,
        0.0000e+00], grad_fn=<SelectBackward0>)


In [14]:
# Predict Text
generation_output = model.generate(input_ids, max_length=50, num_return_sequences=1, do_sample=True, temperature=0.7)

# Decode the generated text
generated_text = tokenizer.decode(generation_output[0], skip_special_tokens=True)
print(f"Generated Text: {generated_text}")

Generated Text: quả cam ngon.

### Không tính đến độ ngon, tôi sẽ đánh giá sự thực sự nhất định về cam tươi ngon vì tôi có thể cảm nhận được hương vị khi nó được làm nóng hoặc được làm lạnh. Bạn có thể cảm
